In [1]:
import pandas as pd
import numpy as np

from pathlib import Path, PurePath

from sklearn.cluster import KMeans
import sys

sys.path.append(str(Path.home()))
sys.path.append(r"C:\Users\LE\Desktop\Jan_uk")

import os
import seaborn as sns
import matplotlib.pyplot as plt

from mix_scripts import mix_api_utilities

In [2]:
#os.chdir(r'C:\Users\bidali.KENYAGRANGE\Documents\Projects\DA\Test')



period = 'Feb 2026'

In [5]:
path = 'mix_report_template_uk.xlsx'
reports_df = pd.read_excel(path, sheet_name='Reports')
reports_df = reports_df[reports_df.to_pull_umbrella > 0]
report_gbs = reports_df.groupby('report_name')
driver_reports = ['Total Kenya HV', 'EABL']

Path('Results').mkdir(exist_ok=True)
for report_name, report_group in report_gbs:
    print(report_name)
    file_name = f'{report_name} Monthly Fleet Report {period}'
    report_result_file_name = Path(f'Results/{report_name}').joinpath(f'{file_name}.xlsx')
    
    if report_result_file_name.exists():
        continue
    
    organisations = report_group.report_dirs.unique()
    organisations = [Path(organisation) for organisation in organisations]
    
    trips_dirs = [organisation.joinpath('Trips') for organisation in organisations]
    events_dirs = [organisation.joinpath('Events') for organisation in organisations]
    
    # Read and filter trip data
    trips_dfs = [mix_api_utilities.read_mix_api_trips(trips_dir) for trips_dir in trips_dirs]
    trips_dfs = [df for df in trips_dfs if not df.empty]
    trips_df_original = pd.concat(trips_dfs) if trips_dfs else pd.DataFrame()
    
    # Read and filter event data
    events_dfs = [mix_api_utilities.read_mix_api_events(events_dir) for events_dir in events_dirs]
    events_dfs = [df for df in events_dfs if not df.empty]
    events_df_original = pd.concat(events_dfs) if events_dfs else pd.DataFrame()
    
    mix_api_utilities.write_bridge_events_df(events_df_original, report_name=report_name)
    events_df = mix_api_utilities.read_bridge_events_df(events_df_original, report_name=report_name)
    trips_df = trips_df_original.copy()
    
    FROM, TO = pd.to_datetime(pd.read_excel(organisations[0].joinpath('report_details.xlsx'), sheet_name='Time', index_col='Name'))
    
    entity_column_name = ['AssetDescription', 'RegistrationNumber']
    
    rag_score = mix_api_utilities.make_rag_score(trips_df, events_df, entity_column_name=entity_column_name)
    utilization = mix_api_utilities.make_daily_utilization(trips_df, FROM, TO)
    fuel_report = mix_api_utilities.make_fuel_report(trips_df)
    
    entity_column_name = 'RegistrationNumber'
    (top_n, top_n2) = mix_api_utilities.write_top_n(events_df, trips_df, entity_column_name=entity_column_name)
    
    path = Path(f'Results/{report_name}')
    path.mkdir(parents=True, exist_ok=True)
    
    with pd.ExcelWriter(path.joinpath(f'{file_name}.xlsx')) as writer:
        rag_score.to_excel(writer, sheet_name='Scoring')
        rag_score.to_excel(writer, sheet_name='Analysis')
        utilization.to_excel(writer, sheet_name='Utilization')
        fuel_report.to_excel(writer, sheet_name='Fuel')
    
    with pd.ExcelWriter(path.joinpath('Top N.xlsx')) as writer2:
        top_n.to_excel(writer2, sheet_name='Top1')
        top_n2.to_excel(writer2, sheet_name='Top2')
    
    distance_col_name = 'DistanceKilometers'
    entity_vehicle_trip = trips_df.pivot_table(values=distance_col_name, index=['ReportName', 'RegistrationNumber'], aggfunc='sum').reset_index()
    entity_vehicle_trip.groupby('ReportName')[distance_col_name].describe().fillna(0).to_excel(path.joinpath('Trips_Summary.xlsx'))
    
    count_col_name = 'TotalOccurances'
    index = ['SiteName', 'DriverName']
    entity_vehicle_event = events_df.pivot_table(values=count_col_name, index=index, aggfunc='sum').reset_index()
    entity_vehicle_event.groupby('SiteName')[count_col_name].describe().fillna(0).to_excel(path.joinpath('Events_Summary2.xlsx'))
    
    sites_pivot_events = events_df.pivot_table(values=count_col_name, columns='SiteName', index='EventName', aggfunc=sum)
    sites_pivot_events.fillna(0).T.to_excel(path.joinpath('Events vs Sites.xlsx'))


Lafarge
Dakawou\Trips
Manara\Trips
Anwarali\Trips
BSP\Trips
Sharlycon Invesment\Trips
Trailink\Trips
Astrum Limited (BSP)\Trips
Fazluna Enterprises\Trips
Hacienda\Trips
Jopeed (BSP)\Trips
Bemoney Investment\Trips
GrandSubterra\Trips
Ponty Pridd\Trips
Arabian Transport\Trips
OML Africa Logistics limited\Trips
Askomint Limited (BSP)\Trips
Eldakk\Trips
Africa Rail Opportunities\Trips
Bikash\Trips
Skipping bridge write. File already exists at: Bridges\Lafarge\bridge_violations.xlsx

